In [51]:
import pandas as pd

df = pd.read_csv('../data/raw/hotel_bookings.csv')
df.shape

(119390, 32)

In [52]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  str    
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  str    
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal                       

In [53]:
df.isnull().sum()

hotel                                  0
is_canceled                            0
lead_time                              0
arrival_date_year                      0
arrival_date_month                     0
arrival_date_week_number               0
arrival_date_day_of_month              0
stays_in_weekend_nights                0
stays_in_week_nights                   0
adults                                 0
children                               4
babies                                 0
meal                                   0
country                              488
market_segment                         0
distribution_channel                   0
is_repeated_guest                      0
previous_cancellations                 0
previous_bookings_not_canceled         0
reserved_room_type                     0
assigned_room_type                     0
booking_changes                        0
deposit_type                           0
agent                              16340
company         

In [54]:
# children : très peu de cas, on suppose 0 enfant par défaut
df['children'] = df['children'].fillna(0)

# agent et company : NaN = "pas d'agence / pas d'entreprise", pas une valeur manquante au sens propre
df['agent'] = df['agent'].fillna(0)
df['company'] = df['company'].fillna(0)

# country : peu de cas, on peut soit remplir avec le pays le plus fréquent, soit marquer "Unknown"
df['country'] = df['country'].fillna('Unknown')

# vérification
df.isnull().sum().sum()  

np.int64(0)

In [55]:
df.duplicated().sum()

np.int64(31994)

## Mise en forme des dates 

In [56]:
df['arrival_date'] = pd.to_datetime(
    df['arrival_date_year'].astype(str) + '-' +
    df['arrival_date_month'] + '-' +
    df['arrival_date_day_of_month'].astype(str),
    format='%Y-%B-%d'
)
df['arrival_date'].head()

0   2015-07-01
1   2015-07-01
2   2015-07-01
3   2015-07-01
4   2015-07-01
Name: arrival_date, dtype: datetime64[us]

## Suppression des doublons

In [57]:
df = df.drop_duplicates()
df.shape

(87396, 33)

In [58]:
df.dtypes

hotel                                        str
is_canceled                                int64
lead_time                                  int64
arrival_date_year                          int64
arrival_date_month                           str
arrival_date_week_number                   int64
arrival_date_day_of_month                  int64
stays_in_weekend_nights                    int64
stays_in_week_nights                       int64
adults                                     int64
children                                 float64
babies                                     int64
meal                                         str
country                                      str
market_segment                               str
distribution_channel                         str
is_repeated_guest                          int64
previous_cancellations                     int64
previous_bookings_not_canceled             int64
reserved_room_type                           str
assigned_room_type  

In [59]:
df['agent'] = df['agent'].astype(int)
df['company'] = df['company'].astype(int)
df['children'] = df['children'].astype(int)

In [60]:
df.dtypes

hotel                                        str
is_canceled                                int64
lead_time                                  int64
arrival_date_year                          int64
arrival_date_month                           str
arrival_date_week_number                   int64
arrival_date_day_of_month                  int64
stays_in_weekend_nights                    int64
stays_in_week_nights                       int64
adults                                     int64
children                                   int64
babies                                     int64
meal                                         str
country                                      str
market_segment                               str
distribution_channel                         str
is_repeated_guest                          int64
previous_cancellations                     int64
previous_bookings_not_canceled             int64
reserved_room_type                           str
assigned_room_type  

### Les valeurs aberantes 

In [61]:
df['adr'].describe()

count    87396.000000
mean       106.337246
std         55.013953
min         -6.380000
25%         72.000000
50%         98.100000
75%        134.000000
max       5400.000000
Name: adr, dtype: float64

In [62]:
df[(df['adults'] == 0) & (df['children'] == 0) & (df['babies'] == 0)].shape

(166, 33)

In [63]:
# Combien de lignes avec adr négatif ou nul ?
df[df['adr'] <= 0].shape

(1779, 33)

In [64]:
# Regarder les lignes avec adr très élevé, pour juger si c'est plausible
df[df['adr'] > 1000][['hotel', 'adr', 'adults', 'stays_in_week_nights', 'stays_in_weekend_nights']]

,hotel,adr,adults,stays_in_week_nights,stays_in_weekend_nights
48515,City Hotel,5400.0,2,1,0


In [65]:
df = df[~((df['adults'] == 0) & (df['children'] == 0) & (df['babies'] == 0))]
df.shape

(87230, 33)

In [66]:
df[df['adr'] < 0].shape
df[df['adr'] == 0].shape

(1643, 33)